In [12]:
import pypsa
import linopy
import pandas as pd

# 1. Δήλωση του path
ipopt_path = r'D:\DEV\Ipopt\bin\ipopt.exe'

# 2. Χειροκίνητη προσθήκη στη λίστα αν δεν υπάρχει ήδη
if 'ipopt' not in linopy.available_solvers:
    linopy.available_solvers.append('ipopt')

n = pypsa.Network()

# Πρόσθεσε αυτό στην αρχή του κώδικα, μετά τη δημιουργία του n
n.add("Carrier", "AC")

n.set_snapshots(["now"])

# 2. Ορισμός Δικτύου (Πιο "στιβαρό" για να βοηθήσουμε τη σύγκλιση του Ipopt)
n.add("Bus", "Bus 1", v_nom=110, v_mag_pu_min=0.9, v_mag_pu_max=1.1)
n.add("Bus", "Bus 2", v_nom=110, v_mag_pu_min=0.9, v_mag_pu_max=1.1)

# Γραμμή με Αντίσταση (r) και Αντίδραση (x) - Τύπος: S = V * I*
n.add("Line", "Line 1-2", bus0="Bus 1", bus1="Bus 2", r=0.001, x=0.01, s_nom=100)


n.add("Load", "Load", bus="Bus 2", p_set=60, q_set=20) # P και Q (Ενεργός/Άεργος)

n.add("Generator", "Gen", bus="Bus 1", p_nom=100, marginal_cost=20)

# --- ΕΠΙΛΟΓΗ 1: LOPF (Γραμμικό - DC) ---
# Μαθηματικά: f = Δθ / x
n.optimize(solver_name='highs')
print(f"LOPF - Παραγωγή: {n.generators_t.p.loc['now', 'Gen']} MW")

# --- ΕΠΙΛΟΓΗ 2: AC OPF (Μη Γραμμικό - AC) ---
# Μαθηματικά: P + jQ = V * (Y* V*)
# Απαιτείται ipopt
# Αντί για n.optimize, χρησιμοποιούμε την ειδική συνάρτηση για AC
# Η n.optimize() της Linopy δυσκολεύεται με μη γραμμικά μοντέλα
try:
    # Προσπάθεια επίλυσης AC OPF με την κλασική μέθοδο του PyPSA
    n.optimize(solver_name='ipopt', solver_io='nl')
except Exception as e:
    print(f"Σφάλμα Linopy: {e}")
    print("Δοκιμή εναλλακτικής μεθόδου...")
    # Εναλλακτικά, αν θέλεις απλώς να δεις αν οι εξισώσεις AC βγάζουν νόημα:
    n.pf() # Αυτό τρέχει ένα απλό Power Flow (όχι optimization)

print(f"AC OPF - Παραγωγή P: {n.generators_t.p.loc['now', 'Gen']} MW")
print(f"AC OPF - Παραγωγή Q: {n.generators_t.q.loc['now', 'Gen']} MVar")
print(f"Τάση στο Bus 2: {n.buses_t.v_mag_pu.loc['now', 'Bus 2']} p.u.")

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 2 primals, 6 duals
Objective: 1.20e+03
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


LOPF - Παραγωγή: 60.0 MW


INFO:linopy.model: Solve problem using Ipopt solver
INFO:linopy.model:Solver options:
 - solver_io: nl
INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001853B779550> for snapshots Index(['now'], dtype='str', name='snapshot')


Σφάλμα Linopy: 'ipopt' is not a valid SolverName
Δοκιμή εναλλακτικής μεθόδου...
AC OPF - Παραγωγή P: nan MW
AC OPF - Παραγωγή Q: 20.003305927384645 MVar
Τάση στο Bus 2: 0.9999785107860545 p.u.
